In [20]:
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import pandas as pd
con = duckdb.connect()
print("DuckDB version:", duckdb.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DuckDB version: 1.3.2


In [21]:
path = "/content/drive/MyDrive/mortgage_denial/hmda_harris_2025.csv"

con.sql(f"""
    SELECT activity_year, county_code, COUNT(*) AS row_count
    FROM '{path}'
    GROUP BY activity_year, county_code
""").df()

,activity_year,county_code,row_count
0,2025,48201,90751


In [22]:
con.sql(f"""
    SELECT action_taken, COUNT(*) AS row_count
    FROM '{path}'
    GROUP BY action_taken
    ORDER BY action_taken
""").df()

,action_taken,row_count
0,1,62641
1,2,3884
2,3,24226


In [23]:
con.sql(f"""
    SELECT loan_purpose,
           COUNT(*) AS applications,
           ROUND(AVG(CASE WHEN action_taken = 3 THEN 1 ELSE 0 END) * 100, 1) AS denial_rate_pct
    FROM '{path}'
    GROUP BY loan_purpose
    ORDER BY applications DESC
""").df()

,loan_purpose,applications,denial_rate_pct
0,1,54854,17.5
1,32,11157,33.3
2,31,10952,31.6
3,4,7963,53.3
4,2,5819,54.6
5,5,6,50.0


In [24]:
con.sql(f"""
    CREATE OR REPLACE TABLE apps AS
    SELECT *
    FROM '{path}'
    WHERE loan_purpose = 1
      AND lien_status = 1
      AND occupancy_type = 1
""")

con.sql("SELECT COUNT(*) AS rows_kept FROM apps").df()

,rows_kept
0,46961


In [25]:
con.sql("""
    SELECT derived_ethnicity,
           COUNT(*) AS applications,
           ROUND(AVG(CASE WHEN action_taken = 3 THEN 1 ELSE 0 END) * 100, 1) AS denial_rate_pct
    FROM apps
    GROUP BY derived_ethnicity
    ORDER BY applications DESC
""").df()

,derived_ethnicity,applications,denial_rate_pct
0,Not Hispanic or Latino,22170,12.7
1,Hispanic or Latino,17023,24.2
2,Ethnicity Not Available,5780,19.6
3,Joint,1982,12.0
4,Free Form Text Only,6,16.7


In [26]:
con.sql("""
    CREATE OR REPLACE TABLE apps2 AS
    SELECT *,
        CASE
            WHEN derived_ethnicity = 'Hispanic or Latino'            THEN 'Hispanic'
            WHEN derived_race = 'White'                                THEN 'Non-Hispanic White'
            WHEN derived_race = 'Black or African American'                                THEN 'Non-Hispanic Black'
            WHEN derived_race = 'Asian'                                THEN 'Non-Hispanic Asian'
            ELSE 'Other / Not available'
        END AS grp,
        CASE WHEN action_taken = 3 THEN 1 ELSE 0 END AS denied
    FROM apps
""")

In [27]:
con.sql("""
    SELECT grp,
           COUNT(*) AS applications,
           ROUND(AVG(denied) * 100, 1) AS denial_rate_pct
    FROM apps2
    GROUP BY grp
    ORDER BY applications DESC
""").df()

,grp,applications,denial_rate_pct
0,Hispanic,17023,24.2
1,Non-Hispanic White,13207,9.4
2,Other / Not available,6810,17.3
3,Non-Hispanic Black,5869,22.0
4,Non-Hispanic Asian,4052,11.9


In [28]:
con.sql("""
    SELECT debt_to_income_ratio, COUNT(*) AS applications
    FROM apps2
    GROUP BY debt_to_income_ratio
    ORDER BY applications DESC
""").df()

,debt_to_income_ratio,applications
0,50%-60%,7018
1,30%-<36%,6039
2,20%-<30%,5245
3,49,2797
4,>60%,2637
5,48,1924
6,44,1860
7,47,1772
8,<20%,1770
9,42,1709


In [29]:
con.sql("""
    CREATE OR REPLACE TABLE apps3 AS
    SELECT *,
        CASE
            WHEN debt_to_income_ratio IN ('<20%', '20%-<30%')                       THEN '1 Under 30%'
            WHEN debt_to_income_ratio = '30%-<36%'                                       THEN '2 30-35%'
            WHEN debt_to_income_ratio IN ('36','37','38','39','40','41','42','43')  THEN '3 36-43%'
            WHEN debt_to_income_ratio IN ('44','45','46','47','48','49')                                      THEN '4 44-49%'
            WHEN debt_to_income_ratio IN ('50%-60%', '>60%')                        THEN '5 50%+'
            ELSE '6 Missing'
        END AS dti_group
    FROM apps2
""")

con.sql("SELECT dti_group, COUNT(*) AS applications FROM apps3 GROUP BY dti_group ORDER BY dti_group").df()

,dti_group,applications
0,1 Under 30%,7015
1,2 30-35%,6039
2,3 36-43%,11767
3,4 44-49%,11683
4,5 50%+,9655
5,6 Missing,802


In [30]:
con.sql("""
    SELECT MIN(TRY_CAST(income AS DOUBLE))    AS min_income,
           MEDIAN(TRY_CAST(income AS DOUBLE)) AS median_income,
           MAX(TRY_CAST(income AS DOUBLE))    AS max_income,
           SUM(CASE WHEN TRY_CAST(income AS DOUBLE) IS NULL
                      OR TRY_CAST(income AS DOUBLE) <= 0 THEN 1 ELSE 0 END) AS bad_income_rows
    FROM apps3
""").df()



,min_income,median_income,max_income,bad_income_rows
0,-401.0,107.0,22842.0,689.0


In [31]:
con.sql("""
    CREATE OR REPLACE TABLE apps4 AS
    SELECT *,
        TRY_CAST(income AS DOUBLE) AS income_num,
        CASE
            WHEN TRY_CAST(income AS DOUBLE) < 50                    THEN '1 Under $50k'
            WHEN TRY_CAST(income AS DOUBLE) < 100                 THEN '2 $50-100k'
            WHEN TRY_CAST(income AS DOUBLE) < 150            THEN '3 $100-150k'
            ELSE '4 $150k+'
        END AS income_group
    FROM apps
    WHERE TRY_CAST(income AS DOUBLE) > 0
""")

con.sql("SELECT income_group, COUNT(*) AS applications FROM apps4 GROUP BY income_group ORDER BY income_group").df()

,income_group,applications
0,1 Under $50k,3046
1,2 $50-100k,17739
2,3 $100-150k,11244
3,4 $150k+,14243


In [32]:
con.sql("""
    CREATE OR REPLACE TABLE apps4 AS
    SELECT *,
        TRY_CAST(income AS DOUBLE) AS income_num,
        CASE
            WHEN TRY_CAST(income AS DOUBLE) < 50   THEN '1 Under $50k'
            WHEN TRY_CAST(income AS DOUBLE) < 100  THEN '2 $50-100k'
            WHEN TRY_CAST(income AS DOUBLE) < 150  THEN '3 $100-150k'
            ELSE '4 $150k+'
        END AS income_group
    FROM apps3
    WHERE TRY_CAST(income AS DOUBLE) > 0
""")

con.sql("SELECT grp, dti_group, income_group, denied FROM apps4 LIMIT 5").df()

,grp,dti_group,income_group,denied
0,Non-Hispanic White,3 36-43%,4 $150k+,0
1,Non-Hispanic Black,4 44-49%,2 $50-100k,1
2,Hispanic,5 50%+,1 Under $50k,1
3,Non-Hispanic White,6 Missing,3 $100-150k,1
4,Non-Hispanic White,1 Under 30%,2 $50-100k,0


In [33]:
con.sql("""
    WITH cells AS (
        SELECT grp, income_group, dti_group,
               COUNT(*)    AS n,
               AVG(denied) AS rate
        FROM apps4
        GROUP BY grp, income_group, dti_group
    ),
    white AS (
        SELECT income_group, dti_group, rate AS white_rate
        FROM cells
        WHERE grp = 'Non-Hispanic White'
    )
    SELECT c.grp,
           SUM(c.n) AS applications,
           ROUND(SUM(c.n * c.rate)       / SUM(c.n) * 100, 1) AS actual_rate,
           ROUND(SUM(c.n * w.white_rate) / SUM(c.n) * 100, 1) AS expected_rate,
           ROUND((SUM(c.n * c.rate) - SUM(c.n * w.white_rate)) / SUM(c.n) * 100, 1) AS unexplained_gap
    FROM cells c
    JOIN white w
      ON c.income_group = w.income_group AND c.dti_group = w.dti_group
    GROUP BY c.grp
    ORDER BY actual_rate DESC
""").df()

,grp,applications,actual_rate,expected_rate,unexplained_gap
0,Hispanic,16782.0,23.6,16.8,6.8
1,Non-Hispanic Black,5791.0,21.4,14.8,6.7
2,Other / Not available,6719.0,16.9,12.3,4.6
3,Non-Hispanic Asian,3973.0,11.2,9.9,1.3
4,Non-Hispanic White,13007.0,9.1,9.1,0.0


In [34]:
con.sql("""
    SELECT grp,
           "denial_reason-1" AS reason,
           COUNT(*) AS denials,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY grp), 1) AS pct_of_group_denials
    FROM apps4
    WHERE denied = 1
    GROUP BY grp, "denial_reason-1"
    ORDER BY grp, denials DESC
""").df()

,grp,reason,denials,pct_of_group_denials
0,Hispanic,1,1454,36.7
1,Hispanic,3,1326,33.4
2,Hispanic,6,413,10.4
3,Hispanic,9,226,5.7
4,Hispanic,4,226,5.7
5,Hispanic,5,148,3.7
6,Hispanic,7,108,2.7
7,Hispanic,2,58,1.5
8,Hispanic,1111,3,0.1
9,Hispanic,8,3,0.1


In [35]:
con.sql("""
    SELECT loan_type, COUNT(*) AS applications,
           ROUND(AVG(denied) * 100, 1) AS denial_rate_pct
    FROM apps4
    GROUP BY loan_type
    ORDER BY loan_type
""").df()

,loan_type,applications,denial_rate_pct
0,1,28681,18.7
1,2,14273,15.3
2,3,3284,12.5
3,4,34,26.5


In [36]:
con.sql("""
    SELECT grp,
           COUNT(*) AS applications,
           ROUND(AVG(CASE WHEN loan_type = 2 THEN 1 ELSE 0 END) * 100, 1) AS pct_fha
    FROM apps4
    GROUP BY grp
    ORDER BY pct_fha DESC
""").df()

,grp,applications,pct_fha
0,Non-Hispanic Black,5791,47.6
1,Hispanic,16782,40.0
2,Other / Not available,6719,27.5
3,Non-Hispanic White,13007,18.9
4,Non-Hispanic Asian,3973,12.9


In [37]:
con.sql("""
    WITH cells AS (
        SELECT grp, income_group, dti_group,
               loan_type,                                   -- NEW 1
               COUNT(*)    AS n,
               AVG(denied) AS rate
        FROM apps4
        GROUP BY grp, income_group, dti_group,
                 loan_type                                  -- NEW 2
    ),
    white AS (
        SELECT income_group, dti_group,
               loan_type,                                   -- NEW 3
               rate AS white_rate
        FROM cells
        WHERE grp = 'Non-Hispanic White'
    )
    SELECT c.grp,
           SUM(c.n) AS applications,
           ROUND(SUM(c.n * c.rate)       / SUM(c.n) * 100, 1) AS actual_rate,
           ROUND(SUM(c.n * w.white_rate) / SUM(c.n) * 100, 1) AS expected_rate,
           ROUND((SUM(c.n * c.rate) - SUM(c.n * w.white_rate)) / SUM(c.n) * 100, 1) AS unexplained_gap
    FROM cells c
    JOIN white w
      ON  c.income_group = w.income_group
      AND c.dti_group    = w.dti_group
      AND c.loan_type    = w.loan_type                     -- NEW 4
    GROUP BY c.grp
    ORDER BY actual_rate DESC
""").df()

,grp,applications,actual_rate,expected_rate,unexplained_gap
0,Hispanic,16778.0,23.6,17.2,6.4
1,Non-Hispanic Black,5788.0,21.4,13.2,8.2
2,Other / Not available,6712.0,16.9,12.5,4.3
3,Non-Hispanic Asian,3973.0,11.2,10.9,0.3
4,Non-Hispanic White,13007.0,9.1,9.1,0.0


In [38]:
folder = "/content/drive/MyDrive/mortgage_denial/"

# 1. Raw denial rate by group
con.sql(f"""
    COPY (
        SELECT grp, COUNT(*) AS applications,
               ROUND(AVG(denied) * 100, 1) AS denial_rate_pct
        FROM apps4
        GROUP BY grp
    ) TO '{folder}result_group_rates.csv' (HEADER)
""")

# 2. Denial rate by DTI bucket
con.sql(f"""
    COPY (
        SELECT dti_group, COUNT(*) AS applications,
               ROUND(AVG(denied) * 100, 1) AS denial_rate_pct
        FROM apps4
        GROUP BY dti_group
        ORDER BY dti_group
    ) TO '{folder}result_dti_rates.csv' (HEADER)
""")

# 3. Credit-history denials per 100 applications, by group
con.sql(f"""
    COPY (
        SELECT grp, COUNT(*) AS applications,
               SUM(CASE WHEN denied = 1 AND "denial_reason-1" = 3 THEN 1 ELSE 0 END) AS credit_history_denials,
               ROUND(SUM(CASE WHEN denied = 1 AND "denial_reason-1" = 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS per_100_apps
        FROM apps4
        GROUP BY grp
    ) TO '{folder}result_credit_history.csv' (HEADER)
""")

# 4. Raw vs. adjusted gap (income + DTI + loan type)
con.sql(f"""
    COPY (
        WITH cells AS (
            SELECT grp, income_group, dti_group, loan_type,
                   COUNT(*) AS n, AVG(denied) AS rate
            FROM apps4
            GROUP BY grp, income_group, dti_group, loan_type
        ),
        white AS (
            SELECT income_group, dti_group, loan_type, rate AS white_rate
            FROM cells WHERE grp = 'Non-Hispanic White'
        )
        SELECT c.grp,
               SUM(c.n) AS applications,
               ROUND(SUM(c.n * c.rate) / SUM(c.n) * 100, 1) AS actual_rate,
               ROUND(SUM(c.n * w.white_rate) / SUM(c.n) * 100, 1) AS expected_rate,
               ROUND((SUM(c.n * c.rate) - SUM(c.n * w.white_rate)) / SUM(c.n) * 100, 1) AS unexplained_gap
        FROM cells c
        JOIN white w ON c.income_group = w.income_group
                    AND c.dti_group = w.dti_group
                    AND c.loan_type = w.loan_type
        GROUP BY c.grp
    ) TO '{folder}result_adjusted_gap.csv' (HEADER)
""")



## Key Findings
- Hispanic applicants were denied at 23.6% and Black applicants at 21.4%, compared with 9.1% for White applicants. That is a raw gap of 14.5 and 12.3 percentage points.
- After controlling for income, debt-to-income (DTI), and loan type, a gap remains: 6.4 points for Hispanic applicants and 8.2 points for Black applicants.
- About 67% of the Black–White gap (8.2 of 12.3) is not explained by these controls, compared with about 44% for Hispanic applicants (6.4 of 14.5).
- Hispanic applicants have the larger raw gap, but more of it is explained by income, DTI, and loan type. Black applicants have the larger unexplained gap.
- Denial rates stay near 10% for DTI between 30% and 49%, then jump to 40.1% once DTI reaches 50% or more.
- DTI was the most common denial reason for every group: 36.7% of Hispanic denials, 39.1% of Black denials, and 32.6% of White denials.
- Credit history was cited 7.9 times per 100 Hispanic applications and 6.3 per 100 Black applications, compared with 2.1 for White applicants, about 3–4 times as often.

## Limits
- Public HMDA data has no credit scores, so the "unexplained" gap likely includes some real credit differences we can't

